[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/05-langchain-retrieval-augmentation.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/05-langchain-retrieval-augmentation.ipynb)

#### [LangChain Handbook](https://www.pinecone.io/learn/series/langchain/)

# Retrieval Augmentation

**L**arge **L**anguage **M**odels (LLMs) have a data freshness problem. Even the most powerful LLMs in the world are not up to date with recent world events.

The world of LLMs is frozen in time. Their world exists as a static snapshot of the world as it was within their training data.

A solution to this problem is *retrieval augmentation*. The idea behind this is that we retrieve relevant information from an external knowledge base and give that information to our LLM. In this notebook we will learn how to do that.

[![Open fast notebook](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/fast-link.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/generation/langchain/handbook/05-langchain-retrieval-augmentation-fast.ipynb)

To begin, we must install the prerequisite libraries that we will be using in this notebook.

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU     datasets==3.6.0     langchain==0.3.25     langchain-community==0.3.25     langchain-google-genai==2.0.10     langchain-qdrant==0.2.1     qdrant-client     sniffio anyio pyparsing     tiktoken==0.9.0

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

Note: you may need to restart the kernel to use updated packages.


## Building the Knowledge Base

In [2]:
from datasets import load_dataset

data = load_dataset(
    "aurelio-ai/reddit-finance",
    split="train",
)
data

Dataset({
    features: ['id', 'subreddit', 'title', 'selftext'],
    num_rows: 107
})

In [3]:
data[5]

{'id': '1k7782l',
 'subreddit': 'stocks',
 'title': 'Waymo reports 250,000 paid robotaxi rides per week in U.S.',
 'selftext': 'Alphabet\xa0reported Thursday that Waymo, its autonomous vehicle unit, is now delivering more than 250,000 paid robotaxi rides per week in the U.S.\n\nCEO Sundar Pichai said Waymo has options in terms of “business models across geographies,” and the robotaxi company is building partnerships with ride-hailing app Uber, automakers and operations and maintenance businesses that tend to its vehicle fleets.\n\n“We can’t possibly do it all ourselves,” said Pichai on a call with analysts for Alphabet’s\xa0first-quarter earnings.\xa0\n\nPichai noted that Waymo has not entirely defined its long-term business model, and there is “future optionality around personal ownership” of vehicles equipped with Waymo’s self-driving technology. The company is also exploring the ways it can scale up its operations, he said.\n\nThe 250,000 paid rides per week are up from 200,000 in F

Many records contain *a lot* of text. Before embedding, we split each article into smaller, more focused **chunks**. Smaller chunks embed more precisely, and at query time retrieval can return just the relevant passage instead of a whole document.

We use `RecursiveCharacterTextSplitter.from_tiktoken_encoder(...)`. The *recursive* part splits on natural boundaries first (paragraphs, then lines, then spaces), while `from_tiktoken_encoder` makes it size chunks by **token count** (via `tiktoken`, fully offline). `tiktoken` is OpenAI's tokenizer rather than Gemini's, but both are sub-word BPE tokenizers so the counts are close — and far better for sizing than raw characters. (The class name still says "Character" because that refers to *how it splits*, not how it *measures*.)

In [4]:
import tiktoken

# tiktoken runs locally (no API calls). Gemini uses a different tokenizer,
# but token counts are close, so this is a good offline approximation for
# sizing chunks - and more accurate than counting characters.
tokenizer = tiktoken.get_encoding("o200k_base")

def tiktoken_len(text):
    return len(tokenizer.encode(text, disallowed_special=()))

tiktoken_len("hello world, this is a chunk of text measured in tokens")

12

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",
    chunk_size=300,      # max tokens per chunk
    chunk_overlap=20,    # tokens shared between consecutive chunks
    separators=["\n\n", "\n", " ", ""]
)

In [6]:
chunks = text_splitter.split_text(data[5]['selftext'])
chunks

['Alphabet\xa0reported Thursday that Waymo, its autonomous vehicle unit, is now delivering more than 250,000 paid robotaxi rides per week in the U.S.\n\nCEO Sundar Pichai said Waymo has options in terms of “business models across geographies,” and the robotaxi company is building partnerships with ride-hailing app Uber, automakers and operations and maintenance businesses that tend to its vehicle fleets.\n\n“We can’t possibly do it all ourselves,” said Pichai on a call with analysts for Alphabet’s\xa0first-quarter earnings.\xa0\n\nPichai noted that Waymo has not entirely defined its long-term business model, and there is “future optionality around personal ownership” of vehicles equipped with Waymo’s self-driving technology. The company is also exploring the ways it can scale up its operations, he said.\n\nThe 250,000 paid rides per week are up from 200,000 in February, before Waymo opened in\xa0Austin\xa0and expanded in the\xa0San Francisco Bay Area\xa0in March.\xa0\n\nWaymo, which is

In [7]:
tiktoken_len(chunks[0]), tiktoken_len(chunks[1])

(279, 291)

Using the `text_splitter` we get much better sized chunks of text. We'll use this functionality during the indexing process later. Now let's take a look at embedding.

## Creating Embeddings

Building embeddings using LangChain's Google Gemini embedding support is fairly straightforward. We first need to add our [Google API key](https://aistudio.google.com/app/apikey) by running the next cell:

In [8]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")     or getpass("Enter your Google API key: ")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

After initializing the API key we can initialize our `gemini-embedding-001` embedding model like so:

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

model_name = 'models/gemini-embedding-001'

embed = GoogleGenerativeAIEmbeddings(
    model=model_name,
    google_api_key=GOOGLE_API_KEY
)

# --- OpenAI alternative (for reference) ---
# from langchain_openai import OpenAIEmbeddings
# embed = OpenAIEmbeddings(
#     model='text-embedding-3-small',
#     openai_api_key=OPENAI_API_KEY
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Now we embed some text like so:

In [10]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

res = embed.embed_documents(texts)
len(res), len(res[0])

(2, 3072)

From this we get *two* (aligning to our two chunks of text) 3072-dimensional embeddings.

Now we move on to initializing our Qdrant vector database.

## Vector Database

We will connect to an existing Qdrant deployment. No API key is required for this deployment, we only need the URL.

In [11]:
# Qdrant here needs no API key, we only need the URL of the running deployment.
QDRANT_URL = "http://109.199.118.73:6333"

Then we create the collection. We will be using Google's `gemini-embedding-001` model for creating the embeddings, so we set the vector `size` to `3072` and use cosine distance.

In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url=QDRANT_URL)
collection_name = "handbook-05-rag"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
    )

# view collection info
client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None)

We should see that the new Qdrant collection has a `points_count` of `0`, as we haven't added any vectors yet.

## Indexing

We can perform the indexing task using the LangChain `QdrantVectorStore` object. The vector store embeds the texts via the Gemini `embed` model automatically and upserts them into the collection. Let's first initialize the vector store.

In [13]:
from langchain_qdrant import QdrantVectorStore

# initialize the vector store object
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

from tqdm.auto import tqdm

batch_limit = 100

texts = []
metadatas = []

for i, record in enumerate(tqdm(data)):
    # first get metadata fields for this record
    url = f"https://reddit.com/r/{record['subreddit']}/comments/{record['id']}"
    metadata = {
        'thread_id': str(record['id']),
        'source': url,
        'subreddit': record['subreddit']
    }
    # now we create chunks from the record text
    record_texts = text_splitter.split_text(record['selftext'])
    # create individual metadata dicts for each chunk
    record_metadatas = [{
        "chunk": j, "text": text, **metadata
    } for j, text in enumerate(record_texts)]
    # append these to current batches
    texts.extend(record_texts)
    metadatas.extend(record_metadatas)
    # if we have reached the batch_limit we can add texts
    if len(texts) >= batch_limit:
        vectorstore.add_texts(texts=texts, metadatas=metadatas)
        texts = []
        metadatas = []

if len(texts) > 0:
    vectorstore.add_texts(texts=texts, metadatas=metadatas)

  0%|          | 0/107 [00:00<?, ?it/s]

We've now indexed everything. We can check the number of vectors in our collection like so:

In [14]:
client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=170, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=Non

## Querying

Now that we've built our collection we can query the vector store directly. The `vectorstore` object we created above is already connected to the Qdrant collection, so we can search it like so:

In [15]:
# The `vectorstore` object created during indexing is already connected to our
# Qdrant collection. If you are reconnecting in a fresh session, you can rebuild
# it from the existing collection like so:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embed,
)

In [16]:
query = "how many robotaxi rides did waymo report in the US?"

vectorstore.similarity_search(
    query,  # our search query
    k=3  # return 3 most relevant docs
)

[Document(metadata={'chunk': 0, 'text': 'Alphabet\xa0reported Thursday that Waymo, its autonomous vehicle unit, is now delivering more than 250,000 paid robotaxi rides per week in the U.S.\n\nCEO Sundar Pichai said Waymo has options in terms of “business models across geographies,” and the robotaxi company is building partnerships with ride-hailing app Uber, automakers and operations and maintenance businesses that tend to its vehicle fleets.\n\n“We can’t possibly do it all ourselves,” said Pichai on a call with analysts for Alphabet’s\xa0first-quarter earnings.\xa0\n\nPichai noted that Waymo has not entirely defined its long-term business model, and there is “future optionality around personal ownership” of vehicles equipped with Waymo’s self-driving technology. The company is also exploring the ways it can scale up its operations, he said.\n\nThe 250,000 paid rides per week are up from 200,000 in February, before Waymo opened in\xa0Austin\xa0and expanded in the\xa0San Francisco Bay A

All of these are good, relevant results. But what can we do with this? There are many tasks, one of the most interesting (and well supported by LangChain) is called _"Retrieval Augmented Generation"_ or RAG.

## Retrieval Augmented Generation

In RAG we take the query as a question that is to be answered by a LLM, but the LLM must answer the question based on the information it is seeing being returned from the `vectorstore`.

To do this we initialize a retrieval pipeline like so:

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# completion llm
llm = ChatGoogleGenerativeAI(
    temperature=0.0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0.0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

# Create prompt template
template = """Answer the question based on the following context:

{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Create LCEL chain
retrieval_chain = (
    {"context": vectorstore.as_retriever(), "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

print(query)

retrieval_chain.invoke(query)

how many robotaxi rides did waymo report in the US?


'Waymo reported delivering more than 250,000 paid robotaxi rides per week in the U.S.'

We can also include the sources of information that the LLM is using to answer our question:

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NL = chr(10)

# Prompt template with source formatting
template = (
    "Answer the question based on the following context. "
    "Include the source URLs in your answer." + NL + NL +
    "{context}" + NL + NL +
    "Question: {question}" + NL + NL +
    "Answer:"
)

prompt = ChatPromptTemplate.from_template(template)

# Create LCEL chain with source formatting
def format_docs(docs):
    return (NL + NL).join(
        f"Source: {doc.metadata.get('source', 'Unknown')}" + NL + doc.page_content
        for doc in docs
    )

retrieval_chain_with_sources = (
    {"context": vectorstore.as_retriever() | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

print("Question:", query)
print(retrieval_chain_with_sources.invoke(query))

Question: how many robotaxi rides did waymo report in the US?


Waymo reported delivering more than 250,000 paid robotaxi rides per week in the U.S. (https://reddit.com/r/stocks/comments/1k7782l, https://www.cnbc.com/2025/04/24/waymo-reports-250000-paid-robotaxi-rides-per-week-in-us.html).


Now we answer the question being asked, *and* return the source of this information being used by the LLM.

Delete the collection to save resources when you're done!

In [19]:
client.delete_collection(collection_name)

True

---